# 🧰 The Python toolbox — every construct this repo is written in

The repo's real code — `models.py`, `fakes.py`, `testing.py` — is built from about a
dozen Python constructs. This notebook teaches each one the same way, every time:
a tiny **toy** you run, then the repo's **real** usage of it, read straight from the
source files with `inspect` (the standard-library tool that shows you a function's or
class's actual code).

**What you'll be able to do after this notebook**

- read any class in this repo and name its parts: attributes, methods, `__init__`, `self`
- explain what a decorator (`@something`) really does — reassignment, not magic
- say precisely what type hints do and *don't* do — and why notebook `02`'s pydantic exists
- recognize custom exceptions, `Enum`s, `with` blocks, dataclasses and daemon threads on sight
- decode `Annotated[str, StringConstraints(...)]` as "a string with a sticky note"

**You need:** `00_orientation.ipynb` (modules, imports, how to run cells).
No infrastructure — everything here runs offline; the real code is *read*, not launched.

**Estimated time:** 60–90 minutes.

> **How to run:** ▶▶ Run All, or step through with **Shift+Enter**. Before each cell,
> predict the output — the markdown above it tells you what to look for.

## 0. Setup: find the repo, meet the cast

First we locate the repo root (so we can point at real files), then import the canonical
example — Ada buys 50 Mbps from Bell, ticket **#7** — from `a2a_interfaces.fixtures`.
Those exact names and numbers appear in every notebook of this course; they live in ONE
place so nothing ever disagrees. Watch for: the repo root's name and ticket **#7**.

In [ ]:
import inspect
from pathlib import Path

# find the repo root from wherever this notebook runs (cwd = e2e/notebooks/course/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())

from a2a_interfaces import fixtures as fx  # the one canonical example (Ada/Bell/#7)

print("repo root       →", ROOT.name)
print("Ada  (buyer)    →", fx.ADA)
print("Bell (provider) →", fx.BELL)
print("ticket id       →", fx.TICKET_ID)
assert (ROOT / "interfaces" / "src" / "a2a_interfaces").exists()

## 1. Classes: factories for objects

Everything you will touch in this repo — an `Offer`, a `FakeChain`, a `ChainClient` — is
an **object**: a bundle of data plus functions that work on that data. The data pieces
are **attributes**; the functions are **methods**; the factory that stamps out these
bundles is a **class**, and each bundle it builds is an **instance**. `__init__` is the
special method that runs once per new instance to store its data.

Let's build the smallest useful class: a ticket with an id and an owner. Watch for:
two instances from one class, each holding its *own* data.

In [ ]:
class Ticket:
    def __init__(self, ticket_id, owner):  # runs once per new instance
        self.ticket_id = ticket_id         # attribute: data stored ON the instance
        self.owner = owner

    def describe(self):                    # method: a function that lives on the class
        return f"ticket #{self.ticket_id}, owned by {self.owner[:8]}…"

t7 = Ticket(fx.TICKET_ID, fx.ADA)          # calling the class runs __init__ → an instance
t8 = Ticket(fx.TELEMETRY_TICKET_ID, fx.ADA)
print("t7 →", t7.describe())
print("t8 →", t8.describe())
assert t7.ticket_id != t8.ticket_id        # separate instances, separate data

Poke it with the three inspection tools you'll use all course: `type()` (which class
built this?), `vars()` (the instance's attributes, as a dict), `dir()` (everything
reachable on it, methods included). Watch for: `describe` shows up in `dir` but **not**
in `vars` — data lives on the instance, methods live on the class.

In [ ]:
print("type(t7) →", type(t7).__name__)
print("vars(t7) →", vars(t7))
print("dir(t7)  →", [name for name in dir(t7) if not name.startswith("_")])

assert "ticket_id" in vars(t7) and "describe" not in vars(t7)
assert "describe" in dir(t7)

The `self` mystery, solved. `t7.describe()` looks like a call with zero arguments — yet
`describe` was defined with one parameter, `self`. That's because `t7.describe()` is
sugar for `Ticket.describe(t7)`: Python passes the instance in as `self` automatically.
Watch for: the two spellings returning the very same string.

In [ ]:
sugar = t7.describe()             # what you normally write
desugared = Ticket.describe(t7)   # what Python actually does
print("t7.describe()       →", sugar)
print("Ticket.describe(t7) →", desugared)
assert sugar == desugared         # `self` IS the instance, passed in for you

**Real repo usage:** `FakeClock` in `e2e/src/e2e/skeleton/fakes.py`. Its *job* — a chain
clock you can fast-forward — is `05_the_walking_skeleton.ipynb`'s story; today we only
read its Python, and it is exactly our `Ticket` pattern: `__init__` stores data, two
methods use it. The leading underscore on `_now` is a convention meaning "internal —
don't touch from outside" (we peek only to learn). Watch for: how little a real,
shipped class needs.

In [ ]:
import e2e.skeleton.fakes as fakes

print(inspect.getsource(fakes.FakeClock))

clock = fakes.FakeClock(fx.WINDOW.start)   # the canonical window start, unix seconds
clock.advance(120)
print("now() after advance(120) →", clock.now())
assert clock.now() == fx.WINDOW.start + 120

**✏️ Your turn 1.1 — predict what lives on the clock**

`clock` was built with `fx.WINDOW.start` and then advanced by 120. Before running,
write your prediction into the TODO: what dict does `vars(clock)` show — which key,
what value? Success: the un-commented assert passes and matches your prediction.

In [ ]:
# TODO — predict BEFORE running:  vars(clock) == { "???": ??? }
print("vars(clock) →", vars(clock))
print("dir has     →", [n for n in dir(clock) if not n.startswith("__")])
# assert vars(clock) == {"_now": fx.WINDOW.start + 120}   # un-comment once your prediction matches

<details><summary>✅ Solution 1.1 — peek only after trying</summary>

```python
assert vars(clock) == {"_now": fx.WINDOW.start + 120}
```

One key only: `advance(120)` mutated the single `_now` attribute — the two methods
never appear in `vars` because data lives on the instance, methods on the class.
</details>

## 2. Exceptions: failures with names

What should a purchase function do when the buyer can't pay? Returning `None` and hoping
someone checks is a bug waiting to happen. Python's answer: **raise an exception** — an
object that aborts the current work and travels up the call stack until someone
**catches** it with `try/except`. If nobody catches it, Python prints the **traceback**
(the path the failure traveled) and stops. Watch for: the line after the failing call
never runs, and the notebook survives because we caught it.

In [ ]:
def buy(price, balance):
    if balance < price:
        raise ValueError("insufficient funds")  # abort — the return below never happens
    return balance - price

try:
    buy(price=10, balance=3)
    print("✗ this line never runs")
except ValueError as err:                       # catch it, keep the notebook alive
    print("✓ caught →", err)
print("life goes on")

`ValueError` says almost nothing — *what* value, *which* rule? Define your **own**
exception: a class inheriting from `Exception`, usually with an empty body, whose *name*
is the message. This is exceptions as API design: the set of named failures a module can
raise tells you, before reading any logic, everything that can go wrong.
Watch for: the class body is just a docstring, and the name arrives with the catch.

In [ ]:
class OutOfBudget(Exception):
    """The buyer cannot pay."""     # the name IS the API; the body is empty

def buy2(price, balance):
    if balance < price:
        raise OutOfBudget(f"need {price}, have {balance}")
    return balance - price

try:
    buy2(price=10, balance=3)
except OutOfBudget as err:
    print("✓ caught →", type(err).__name__, "—", err)

**Real repo usage:** `e2e/src/e2e/skeleton/fakes.py` defines four named failures its fake
chain raises — each mirrors a refusal the real smart contract makes (you'll trigger those
for real in `07_chainmcp_the_signing_adapter.ipynb`). Watch for: how much the four names
*alone* tell you about what can go wrong in a sale.

In [ ]:
for exc in (fakes.OfferAlreadyUsed, fakes.OfferExpired,
            fakes.WrongConsumer, fakes.InsufficientFunds):
    print(f"{exc.__name__:<18} → {exc.__doc__}")
    assert issubclass(exc, Exception)

`except` selects **by type** — an `except OfferExpired` handler lets an
`OfferAlreadyUsed` fly right past. That is the point of naming failures: a caller can
handle a replayed offer differently from an expired one. Watch for: the inner (wrong)
handler staying silent while the outer (right) one catches.

In [ ]:
def punch_salt(salt, consumed):
    if salt in consumed:
        raise fakes.OfferAlreadyUsed(salt)
    consumed.add(salt)

consumed = {fx.SALT}                       # the canonical salt: already spent
try:
    try:
        punch_salt(fx.SALT, consumed)
    except fakes.OfferExpired:             # wrong type: does NOT catch
        print("✗ this handler never fires")
except fakes.OfferAlreadyUsed as err:      # right type: catches
    print("✓ caught OfferAlreadyUsed →", str(err)[:12], "…")

**✏️ Your turn 2.1 — replay a salt and read the refusal**

`punch_salt` accepts `fresh_salt` once. Do the TODO — punch the *same* salt a second
time — and read which named failure fires. Success: the printed exception name alone
tells you which rule you broke.

In [ ]:
fresh_consumed = set()
fresh_salt = "0x" + "07" * 32          # a salt nobody has spent yet

punch_salt(fresh_salt, fresh_consumed)
print("first punch  → ✓ accepted")

caught = None
try:
    pass  # TODO: replace with a SECOND punch_salt(fresh_salt, fresh_consumed)
except fakes.OfferAlreadyUsed as err:
    caught = type(err).__name__
print("second punch →", caught)
# assert caught == "OfferAlreadyUsed"   # un-comment once the replay is refused

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
punch_salt(fresh_salt, fresh_consumed)   # the second punch, inside the try
assert caught == "OfferAlreadyUsed"
```

One salt = one sale: the second punch is exactly the replay the real contract refuses,
and the exception's *name* carries the whole diagnosis.
</details>

## 3. f-strings: talking about your data

Every print in this course uses them. An **f-string** is a string literal with a leading
`f` where `{...}` holes evaluate live Python. After the value, an optional `:format-spec`
controls rendering. Watch for the specs: `,` thousands separators, `.0f` no decimals,
`x` hexadecimal, and `064x` — hex, zero-padded to 64 characters.

In [ ]:
capacity = fx.CAPACITY_50_MBPS  # 50_000_000 bits per second (underscores group digits)
print(f"capacity → {capacity} bps")                    # plain hole
print(f"readable → {capacity:,} bps")                  # , = thousands separators
print(f"in Mbps  → {capacity / 1_000_000:.0f} Mbps")   # .0f = no decimals
print(f"as hex   → {capacity:x}")                      # x = hexadecimal (base 16)
print(f"padded   → {fx.TICKET_ID:064x}")               # 064x = hex, zero-padded to 64

**Real repo usage:** `fixtures.py` builds ticket #7's on-chain `resource_id` with exactly
that last spec — 64 hex characters = 32 bytes, one full chain word. (Dissecting such
blobs byte-by-byte is `04_the_canonical_example.ipynb`'s job.) Watch for: our hand-rolled
string matching the shipped fixture, character for character.

In [ ]:
fx_lines = inspect.getsource(fx).splitlines()
print("the real line →", next(l for l in fx_lines if l.startswith("RESOURCE_ID")))

ours = "0x" + f"{fx.TICKET_ID:064x}"
print("rebuilt       →", ours)
assert ours == fx.RESOURCE_ID   # same spec, same bytes — one source of truth

**✏️ Your turn 3.1 — rebuild ticket #8's resource_id**

The telemetry sale (ticket **#8**) gets its on-chain id from the very same `064x`
recipe. Change one value in the scaffold, re-run, and watch the last hex character
flip. Success: the un-commented assert passes against the shipped fixture.

In [ ]:
rebuilt = "0x" + f"{fx.TICKET_ID:064x}"   # TODO: swap in fx.TELEMETRY_TICKET_ID
print("rebuilt →", rebuilt)
print("shipped →", fx.TELEMETRY_RESOURCE_ID)
# assert rebuilt == fx.TELEMETRY_RESOURCE_ID   # un-comment once they match

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
rebuilt = "0x" + f"{fx.TELEMETRY_TICKET_ID:064x}"
assert rebuilt == fx.TELEMETRY_RESOURCE_ID
```

One spec, two tickets — only the number in the hole changes (`…07` → `…08`); both
shipped resource ids come from this same f-string recipe.
</details>

## 4. Type hints: labels, not laws

Almost every function in this repo is written like
`def fulfill(self, signed: SignedOffer, buyer: str) -> int`. The `: str` and `-> int`
parts are **type hints** — declarations of what *should* flow in and out.

Here's the part that shocks everyone the first time: **Python does not enforce them.**
Watch this function, hinted int-only, cheerfully accept a string.

In [ ]:
def double(x: int) -> int:   # ": int" promises… nothing, it turns out
    return x * 2

print("double(21)   →", double(21))
print("double('ha') →", double("ha"), "  ← a STRING sailed through the int hint")
assert double("ha") == "haha"   # no error, no warning — hints are labels, not laws

So why does the repo hint everything? Because the labels are *stored and readable*: your
editor reads them to autocomplete, type-checkers like `mypy` read them to catch bugs
before running — and, the reason this project cares most, **pydantic reads them and turns
them into actual enforcement** at every package border (`02_pydantic_from_zero.ipynb` is
exactly that story). Watch for: the hints sitting in a plain dict on the function — and
note the *quotes* around the repo function's hints. Hold that thought for section 13.

In [ ]:
print("toy hints, stored →", double.__annotations__)
print("repo signature    →", inspect.signature(fakes.FakeChain.fulfill))

ann = fakes.FakeChain.fulfill.__annotations__
print("repo hints        →", ann)
assert ann["buyer"] == "str" and ann["return"] == "int"  # strings! (section 13 explains)

The hint vocabulary, part 1 — containers. `list[str]` = a list of strings;
`dict[str, Any]` = a dict with string keys and values of *any* type (`Any` is the
"no promises" hint). Watch for: the repo's real fields using exactly these, in
`interfaces/src/a2a_interfaces/models.py`.

In [ ]:
from typing import Any
from a2a_interfaces import models

def label_paths(paths: list[str]) -> dict[str, Any]:
    return {"paths": paths, "count": len(paths)}

print("toy  →", label_paths(["/interface[name=ethernet-1/1]/statistics"]))

MODEL_SRC = inspect.getsource(models).splitlines()   # kept for later sections
print("real →", next(l.strip() for l in MODEL_SRC if l.strip().startswith("sensor_paths:")))
print("real →", next(l.strip() for l in MODEL_SRC if l.strip().startswith("terms_doc:")))

Part 2 — either/or. `Union[int, str]` means "an int OR a str"; modern Python spells it
`int | str`. `Optional[str]` is nothing more than `Union[str, None]` — "a string, or
nothing". The repo writes the `|` form everywhere (e.g. `deployment: dict | None = None`
in chainmcp). Watch for: the assert proving all three spellings are one meaning.

In [ ]:
from typing import Optional, Union

def find_owner(ticket_id: int) -> str | None:   # modern | spelling
    return fx.ADA if ticket_id == fx.TICKET_ID else None

print("known ticket   →", find_owner(fx.TICKET_ID))
print("unknown ticket →", find_owner(999))

assert Optional[str] == Union[str, None] == (str | None)  # three spellings, one meaning

Part 3 — `Literal`: not "a string" but "**exactly this** string". In plain Python it is
still just a label (watch the banana go through) — but in `02` pydantic turns `Literal`
fields into real gates, and the repo uses one as the tag that tells a bandwidth need
apart from a telemetry need. Watch for: the two real `kind:` field lines.

In [ ]:
from typing import Literal

def set_state(state: Literal["active", "torn_down"]) -> str:
    return f"state = {state}"

print(set_state("active"))
print(set_state("banana"), "  ← plain Python: still not enforced")

for line in MODEL_SRC:
    if line.strip().startswith("kind: Literal"):
        print("real →", line.strip())

**✏️ Your turn 4.1 — write a Literal-tagged function of your own**

Mirror the repo's `kind:` tags: finish `label_need` so it describes its argument, e.g.
`"a bandwidth need"`. Then look at the off-menu call — plain Python still lets
`"banana"` through. Success: both asserts pass after your one-line edit.

In [ ]:
def label_need(kind: Literal["bandwidth", "telemetry"]) -> str:
    return "a ??? need"   # TODO: use an f-string hole for `kind`

print("on-menu  →", label_need("bandwidth"))
print("off-menu →", label_need("banana"), " ← labels, not laws (until 02)")
# assert label_need("telemetry") == "a telemetry need"   # un-comment once written
# assert label_need("banana") == "a banana need"         # off-menu STILL passes — that's the point

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
def label_need(kind: Literal["bandwidth", "telemetry"]) -> str:
    return f"a {kind} need"
```

Same spelling as the repo's real `kind:` fields — and just as unenforced here;
pydantic (notebook `02`) is what turns this `Literal` into an actual gate.
</details>

## 5. Type aliases: a name for a type

The repo needs "an int that fits the chain's 64-bit width" in a dozen places. Writing
the full hint each time is noise — and a drift risk when it changes. A **type alias** is
just an assignment: the name on the left IS the type on the right; nothing new is
created. Watch for: `Meters is float` → True — the very same object.

In [ ]:
Meters = float                    # an alias: a new NAME, not a new type

def wingspan(w: Meters) -> Meters:
    return w * 2

print("wingspan(0.7)   →", wingspan(0.7))
print("Meters is float →", Meters is float)
assert Meters is float

**Real repo usage:** the top of `interfaces/src/a2a_interfaces/models.py` defines the
repo's number vocabulary once — `Uint8`, `Uint32`, `Uint64`, bounded to the chain's
integer widths. Read them now; the `Annotated[...]` wrapper they use is the *next*
section's whole topic. Watch for: `2**64 - 1`, the largest value a chain uint64 holds.

In [ ]:
start = next(i for i, l in enumerate(MODEL_SRC) if l.startswith("Uint8"))
print("\n".join(MODEL_SRC[start : start + 3]))

from a2a_interfaces.models import Uint64
print("\nUint64 is →", Uint64)
assert 2**64 - 1 == 18_446_744_073_709_551_615   # the ceiling the chain enforces

**✏️ Your turn 5.1 — coin an alias of your own**

`Bps` below names the unit `fx.CAPACITY_50_MBPS` is measured in. Finish `to_mbps` so
it converts bits per second to Mbps. Success: the conversion prints `50.0` and both
asserts pass — including the one proving your alias created no new type.

In [ ]:
Bps = int      # your alias: bits per second, the unit fx.CAPACITY_50_MBPS is in

def to_mbps(rate: Bps) -> float:
    return 0.0   # TODO: convert — divide by 1_000_000

print("to_mbps(fx.CAPACITY_50_MBPS) →", to_mbps(fx.CAPACITY_50_MBPS))
print("Bps is int                   →", Bps is int)
# assert to_mbps(fx.CAPACITY_50_MBPS) == 50.0   # un-comment once converted
# assert Bps is int                             # the alias made no new type

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
def to_mbps(rate: Bps) -> float:
    return rate / 1_000_000
```

`Bps` is only a readable name — `Bps is int` stays True, the same way `Uint64` is one
name for "an int, bounded" rather than a new type.
</details>

One deliberate break: the aliases are internal building blocks, so the package's front
door refuses to export them — `from a2a_interfaces import Uint64` fails on purpose (you
reach them via `a2a_interfaces.models`). Watch for: the `ImportError` naming exactly
what you asked for.

In [ ]:
try:
    from a2a_interfaces import Uint64 as _
    print("✗ should not have imported")
except ImportError as err:
    print("✓ front door refused →", err)

## 6. `Annotated`: a sticky note on a type

`Annotated[X, note]` is still just `X` as far as Python is concerned — the note is a
sticky attached for *whoever chooses to look*. Plain Python never looks. Watch for:
`get_args` peeling type and note apart, and the negative number sailing through
unchecked.

In [ ]:
from typing import Annotated, get_args

PositiveInt = Annotated[int, "must be > 0"]   # an int, wearing a sticky note

def stock_level(n: PositiveInt) -> str:
    return f"{n} in stock"

print(stock_level(-5), "  ← plain Python never read the note")
print("get_args →", get_args(PositiveInt))
assert get_args(PositiveInt) == (int, "must be > 0")

**Real repo usage:** `models.py`'s hex-string aliases put *machine-readable* sticky notes
— `StringConstraints(pattern=...)` — on plain `str`. An **Address** must be exactly 40
hex characters (20 bytes); a **Signature** 130 (65 bytes). Pydantic is the reader of
these notes: in `02` you'll watch it reject `"0x123"` using exactly this pattern. Today
we only read them. Watch for: the regex inside the note when we unwrap `Address`.

In [ ]:
start = next(i for i, l in enumerate(MODEL_SRC) if l.startswith("Address"))
end = next(i for i, l in enumerate(MODEL_SRC) if l.startswith("DecimalString"))
print("\n".join(MODEL_SRC[start : end + 1]))

base_type, note = get_args(models.Address)
print("\nAddress unwraps to →", base_type.__name__, "+ a note with pattern:", note.pattern)
assert base_type is str and "{40}" in note.pattern

**✏️ Your turn 6.1 — predict Signature's sticky note**

The section said a **Signature** is 65 bytes of hex. Predict the `{N}` count inside
`models.Signature`'s pattern BEFORE running, write it into the TODO, then unwrap and
check. Success: the un-commented assert confirms your N.

In [ ]:
# TODO — predict BEFORE running:  the pattern ends in {N}$ — N = ???
sig_type, sig_note = get_args(models.Signature)
print("Signature unwraps to →", sig_type.__name__, "+ pattern:", sig_note.pattern)
# assert "{130}" in sig_note.pattern   # un-comment once your prediction is confirmed

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
assert "{130}" in sig_note.pattern   # 65 bytes × 2 hex chars per byte = 130
```

Same anatomy as `Address`, longer ruler — one hex character per half-byte, so 65
bytes need 130 of them.
</details>

## 7. Decorators, from absolute zero

The repo's files carry `@dataclass`, `@property`, `@runtime_checkable`, `@app.post`. The
`@` looks like magic; it is three ordinary facts stacked on each other.

**Fact 1: functions are objects.** You can assign one to another name and pass it around
like any value — *without* parentheses, which would call it. Watch for: two names, one
function.

In [ ]:
def greet():
    return "hello"

also_greet = greet   # no parentheses: handing over the function itself, not its result
print("also_greet()   →", also_greet())
print("same object?   →", also_greet is greet)
print("its name       →", greet.__name__)
assert also_greet is greet

**Fact 2: a function can take a function as an argument** — and call it, without knowing
in advance which function it will be handed. Watch for: `twice` never mentions `greet`.

In [ ]:
def twice(func):            # func is a function — just another argument
    return [func(), func()]

print("twice(greet) →", twice(greet))
assert twice(greet) == ["hello", "hello"]

**Fact 3: a function can build and return a NEW function.** Combine all three: take a
function in, define a `wrapper` that calls it and changes something, hand the wrapper
back. That wrapper pattern IS what every decorator is. Watch for: `loud` is a
*different* function than `greet`.

In [ ]:
def shout(func):
    def wrapper():
        return func().upper() + "!"   # call the original, then embellish
    return wrapper                    # hand back the NEW function

loud = shout(greet)
print("loud()         →", loud())
print("loud is greet? →", loud is greet)
assert loud() == "HELLO!"

Now the `@` syntax: writing `@shout` above a `def` means exactly
`greet2 = shout(greet2)` — define, transform, **reassign**, in one gesture. Proof that
the reassignment really happened: the decorated function's `__name__` is `'wrapper'`.
Watch for that tell.

In [ ]:
@shout                      # sugar for: greet2 = shout(greet2)
def greet2():
    return "hello"

print("greet2()        →", greet2())
print("greet2.__name__ →", greet2.__name__, "  ← the wrapper replaced the original")
assert greet2() == "HELLO!" and greet2.__name__ == "wrapper"

**✏️ Your turn 7.1 — break the wrapper's assumption**

`shout`'s wrapper calls `.upper()` on whatever the wrapped function returns — here it
decorates one that returns ticket #7's *number*, and the assumption explodes. Success:
you can name (in the TODO comment) the exact expression inside `wrapper` that raised.

In [ ]:
@shout                        # wrapper assumes func() returns a string…
def ticket_number():
    return fx.TICKET_ID       # …but this returns an int

caught = None
try:
    ticket_number()
except Exception as e:
    caught = e
print("→", type(caught).__name__, ":", caught)
# TODO: which expression inside shout's wrapper raised?  answer: ...
# assert type(caught).__name__ == "AttributeError"   # un-comment once you've named it

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
# func().upper() raised — func() returned 7, and ints have no .upper()
assert type(caught).__name__ == "AttributeError"
```

The decorator machinery worked perfectly — the failure is *inside* the wrapper, at
call time, exactly where the traceback points.
</details>

That's the whole trick. Now a census of the decorators this repo actually ships, with
one-line roles — each gets its deep dive in a later notebook:

| decorator | role | deep dive |
|---|---|---|
| `@dataclass` | writes `__init__`/`__repr__` for you | next section |
| `@runtime_checkable` | lets `isinstance()` test a Protocol | `03_protocols_and_ports.ipynb` |
| `@property` | method you read like an attribute (no parens) | below, and `07_chainmcp_the_signing_adapter.ipynb` |
| `@app.post(...)` | registers a function as an HTTP endpoint | `08_controller_the_bouncer.ipynb` |

Watch for: how *few* decorators three real modules use, and that you now recognize
every one as "a function transforming what's beneath it".

In [ ]:
from a2a_interfaces import ports
import chainmcp.testing as testing
import chainmcp.client as chain_client

for mod in (ports, testing, chain_client):
    hits = [l.strip() for l in inspect.getsource(mod).splitlines() if l.strip().startswith("@")]
    print(f"{mod.__name__:<18} → {hits}")

`@property` in sixty seconds — it turns a method into a *computed attribute*: callers
write `client.address` with no parentheses, and the method runs. The real one,
`ChainClient.address`, derives the agent's public address from its private key on
demand — callers get the address, **never the key** (rule 2's custody discipline; the
full story is `07_chainmcp_the_signing_adapter.ipynb`). Watch for: no `()` at either
call site.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def diameter(self):        # computed on read — no () for the caller
        return self.radius * 2

print("Circle(2).diameter →", Circle(2).diameter, "  ← no parentheses")
assert Circle(2).diameter == 4

print("\nthe real one (chainmcp/src/chainmcp/client.py):")
print(inspect.getsource(chain_client.ChainClient.address.fget))

## 8. Dataclasses: a decorator that writes your `__init__`

You just learned that decorators transform functions. `@dataclass` transforms a *class*:
you list the attributes as type-hinted names, and it generates `__init__` and a readable
`__repr__` for you. Compare by hand. Watch for: the plain class's useless default
`<...object at 0x...>` repr versus the dataclass's readable one.

In [ ]:
from dataclasses import dataclass

class PlainTicket:                        # by hand: boilerplate
    def __init__(self, ticket_id, owner):
        self.ticket_id = ticket_id
        self.owner = owner

@dataclass
class DataTicket:                         # generated: __init__ AND __repr__
    ticket_id: int
    owner: str

print("plain →", PlainTicket(fx.TICKET_ID, fx.ADA))
print("data  →", DataTicket(fx.TICKET_ID, fx.ADA))

**Real repo usage:** `AnvilChain` in `chainmcp/src/chainmcp/testing.py` — the handle you
get back when a disposable test blockchain is launched (you'll launch one for real in
`06_blockchain_from_zero.ipynb`). Three data fields with generated plumbing, plus two
hand-written methods. Watch for: fields declared as bare annotated names — no `__init__`
in sight, yet the class is constructible.

In [ ]:
print(inspect.getsource(testing.AnvilChain))

field_names = list(testing.AnvilChain.__dataclass_fields__)
print("generated fields →", field_names)
assert field_names == ["process", "rpc_url", "deployment"]

**✏️ Your turn 8.1 — make the generated `__init__` complain**

The scaffold builds an `AnvilChain` with only two of its three fields. Predict which
exception fires and which field the message names, write it into the TODO, then run
and read. Success: the message names the exact field you left out.

In [ ]:
caught = None
try:
    # TODO — predict BEFORE running: which exception, naming which missing field?
    half_built = testing.AnvilChain(process=None, rpc_url="http://127.0.0.1:8545")
except TypeError as e:
    caught = e
print("→", type(caught).__name__, ":", caught)
# assert "deployment" in str(caught)   # un-comment once you've read the message

<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
assert "deployment" in str(caught)   # TypeError: missing 1 required positional argument
```

Proof the decorator really wrote an `__init__`: nobody hand-coded that error — it
comes from the generated constructor, which demands every declared field.
</details>

The catch — and the bridge to `02`: a dataclass **trusts** whatever you pass. Its field
hints are labels here too (section 4!). We can build a nonsense `AnvilChain` — a string
for a process, a number for a URL — and nothing objects. That's acceptable for internal
plumbing built by code you control; it is NOT acceptable for data arriving from another
agent over the network. That border is where pydantic enforces (`02`: *dataclass trusts,
pydantic checks*). Watch for: no error at all.

In [ ]:
bogus = testing.AnvilChain(process="not a process", rpc_url=123, deployment=None)
print("dataclass accepted →", bogus)
print("rpc_url type       →", type(bogus.rpc_url).__name__,
      "  ← an int went through the str hint")
assert bogus.rpc_url == 123   # dataclass trusts; pydantic (notebook 02) checks

## 9. Inheritance in ten lines

A class can be built *on top of* another: `class Dog(Animal)` means Dog **inherits**
everything Animal has, and may add or override pieces. You've already used it — every
custom exception was `class X(Exception)`. Two later notebooks need it: `02`'s models
all inherit from one strict base, and `06`'s Solidity contract inherits ERC721 (yes,
Solidity has it too). Watch for: `bark` exists only on Dog, `eat` is inherited
unchanged, `speak` is overridden.

In [ ]:
class Animal:
    def speak(self):
        return "…"
    def eat(self):
        return "nom"

class Dog(Animal):        # Dog IS an Animal, plus/minus
    def speak(self):      # override: replace the parent's version
        return "woof"
    def bark(self):       # extend: brand-new method
        return "WOOF!"

rex = Dog()
print("speak →", rex.speak(), "| eat →", rex.eat(), "| bark →", rex.bark())
assert isinstance(rex, Animal) and rex.eat() == "nom" and rex.speak() == "woof"

**✏️ Your turn 9.1 — give Cat a voice**

`Cat` inherits everything and overrides nothing, so it speaks Animal's `"…"`. Add a
`speak` override returning `"meow"`, re-run, and note what changed — and what didn't.
Success: `speak` is yours, `eat` is still inherited, both asserts pass.

In [ ]:
class Cat(Animal):
    pass   # TODO: override speak() to return "meow"

felix = Cat()
print("felix.speak() →", felix.speak(), " ← Animal's version until you override")
print("felix.eat()   →", felix.eat())
# assert felix.speak() == "meow" and felix.eat() == "nom"   # un-comment after your override

<details><summary>✅ Solution 9.1 — peek only after trying</summary>

```python
class Cat(Animal):
    def speak(self):
        return "meow"
```

Overriding replaces only what you redefine — `eat` still resolves up the MRO to
`Animal`, the same lookup that sends `BandwidthNeed`'s missing methods up to `BaseModel`.
</details>

**Real repo usage:** ask any repo class for its **MRO** (method resolution order — the
chain of parents Python searches, in order). `BandwidthNeed` inherits from an internal
`_Frozen` base, which inherits from pydantic's `BaseModel` — *why* the repo does that is
the heart of `02_pydantic_from_zero.ipynb`. Watch for: both ancestries ending at
`object`, the ancestor of everything in Python.

In [ ]:
need_parents = [c.__name__ for c in models.BandwidthNeed.__mro__]
print("BandwidthNeed ancestry →", " → ".join(need_parents))
print("OfferExpired ancestry  →",
      " → ".join(c.__name__ for c in fakes.OfferExpired.__mro__))
assert need_parents == ["BandwidthNeed", "_Frozen", "BaseModel", "object"]

## 10. `Enum`: a closed menu of values

A session's state should be one of five words, *ever*. Bare strings invite `"activ"`
typos that fail silently, far from the cause. An **Enum** is a class whose instances are
a fixed, named menu; anything off-menu is rejected immediately at the border. Watch for:
the round-trip by value, and the off-menu rejection.

In [ ]:
from enum import Enum

class Color(Enum):
    RED = "red"
    GREEN = "green"

print("members  →", [c.name for c in Color])
print("by value →", Color("red"))
try:
    Color("purple")
except ValueError as err:
    print("off-menu → ✓ rejected:", err)

But plain Enum members are NOT their values: `Color.RED == "red"` is `False`, and JSON —
the text format every payload in this project travels as (properly introduced in `02`) —
refuses them outright. The repo's states must travel in JSON, so it mixes `str` in:
`class SessionState(str, Enum)` makes each member a *real string* as well as an enum
member. Watch for: the real source, then three checks that all come up string-friendly.

In [ ]:
import json
from a2a_interfaces.models import SessionState

print(inspect.getsource(SessionState))
print("ACTIVE == 'active' →", SessionState.ACTIVE == "active")
print("is a str?          →", isinstance(SessionState.ACTIVE, str))
print("json.dumps         →", json.dumps(SessionState.ACTIVE))
assert SessionState.ACTIVE == "active" and json.dumps(SessionState.ACTIVE) == '"active"'

**✏️ Your turn 10.1 — build a JSON-ready enum yourself**

The repo sells exactly two service kinds. Finish `ServiceKind` with `SessionState`'s
`str` mixin recipe: add the missing `TELEMETRY = "telemetry"` member. Success: both
asserts pass — round-trip by value AND clean `json.dumps`.

In [ ]:
class ServiceKind(str, Enum):
    BANDWIDTH = "bandwidth"
    # TODO: add the other thing the repo sells: TELEMETRY = "telemetry"

print("members    →", [m.value for m in ServiceKind])
print("json-ready →", json.dumps(ServiceKind.BANDWIDTH))
# assert ServiceKind("telemetry") is ServiceKind.TELEMETRY      # un-comment once added
# assert json.dumps(ServiceKind.TELEMETRY) == '"telemetry"'

<details><summary>✅ Solution 10.1 — peek only after trying</summary>

```python
class ServiceKind(str, Enum):
    BANDWIDTH = "bandwidth"
    TELEMETRY = "telemetry"
```

The `str` mixin is the whole trick — each member IS a real string, so JSON borders
(and `08`'s controller states) never choke on it.
</details>

Break it on purpose: feed the plain `Color` enum to JSON and watch it refuse — the `str`
mixin is the difference between a state that can travel and one that can't. Then meet
the repo's other enum: `ErrorCode`, the controller's complete deny vocabulary (every one
of the ten gets exercised in `08_controller_the_bouncer.ipynb`). Watch for: the
`TypeError`, then exactly ten codes.

In [ ]:
from a2a_interfaces.models import ErrorCode

try:
    json.dumps(Color.RED)
except TypeError as err:
    print("plain Enum → ✗ JSON refuses:", err)

print("\nthe controller's deny vocabulary:")
for code in ErrorCode:
    print("  ", code.value)
assert len(list(ErrorCode)) == 10 and Color.RED != "red"

## 11. Context managers: `with` = borrow, then guaranteed return

Sockets, files, and database handles must be returned to the operating system *even when
your code blows up mid-use*. `try/finally` works but is noisy. A **context manager**
packages the pattern: `with X() as x:` calls `X.__enter__` on the way in and —
guaranteed — `X.__exit__` on the way out, exception or not. Watch for the order:
opened → inside → closed.

In [ ]:
class Door:
    def __enter__(self):
        print("1. opened")
        return self
    def __exit__(self, exc_type, exc, tb):
        print("3. closed  ← guaranteed, even on error")
        return False          # False = don't swallow exceptions

with Door():
    print("2. inside")

Proof of the guarantee: raise *inside* the `with` block and watch `__exit__` still run
before the exception continues outward. Watch for: "closed" printing BEFORE the caught
error.

In [ ]:
try:
    with Door():
        raise RuntimeError("boom mid-use")
except RuntimeError as err:
    print("4. caught outside →", err)

**✏️ Your turn 11.1 — predict the closing order**

Two doors, nested. Before running, write into the TODO which door closes FIRST —
inner or outer. Success: the `order` list ends with your predicted sequence and the
un-commented assert passes.

In [ ]:
order = []

class NamedDoor:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        order.append(f"opened {self.name}"); print("→", order[-1]); return self
    def __exit__(self, exc_type, exc, tb):
        order.append(f"closed {self.name}"); print("→", order[-1]); return False

# TODO — predict BEFORE running: which closes first, inner or outer?  answer: ...
with NamedDoor("outer"), NamedDoor("inner"):
    print("→ both open")
# assert order[-2:] == ["closed inner", "closed outer"]   # un-comment once confirmed

<details><summary>✅ Solution 11.1 — peek only after trying</summary>

```python
assert order[-2:] == ["closed inner", "closed outer"]
```

`with` blocks unwind like a stack — last opened, first closed — which is why nested
borrowed resources (a socket inside a chain inside a test) always release in a safe order.
</details>

**Real repo usage:** `_free_port()` in `chainmcp/src/chainmcp/testing.py` (leading
underscore: internal — we peek to learn). It borrows a socket inside a `with`, binds to
port `0` — the OS convention for "you pick a free one" — reads back which port it got,
and the `with` guarantees the socket is returned. The disposable-chain launcher uses it
so two notebooks never fight over a port. It needs zero infrastructure, so call it live.
Watch for: a (very likely) different port every run.

In [ ]:
print(inspect.getsource(testing._free_port))

port = testing._free_port()
print("the OS handed us port →", port)
assert 1 <= port <= 65535

## 12. Threads: doing two things at once (a preview)

In `07`, a chain client must *watch* for revocation events while the rest of the program
keeps working. A **thread** is a second line of execution inside the same program;
`daemon=True` marks it a background helper that must never keep the program alive on its
own. Six lines: start one, wait for it with `join()`, inspect its work. Watch for: the
note was appended by the *background* thread, not this cell.

In [ ]:
import threading

notes = []
worker = threading.Thread(target=lambda: notes.append("ran in the background"), daemon=True)
worker.start()   # begins running alongside this cell
worker.join()    # wait here until it finishes
print("notes →", notes)
assert notes == ["ran in the background"]

**Real repo usage:** `ChainClient.watch_revoked` starts exactly such a thread — its
target is a polling loop, `daemon=True` so a finished notebook is never held hostage by
its watcher. Today you only need to *recognize* the construct; the delivery semantics
are `07_chainmcp_the_signing_adapter.ipynb`'s story. Watch for: the same two calls you
just made — `Thread(...)`, then `.start()`.

In [ ]:
watch_src = inspect.getsource(chain_client.ChainClient.watch_revoked).splitlines()
i = next(n for n, l in enumerate(watch_src) if "threading.Thread" in l)
print("chainmcp/src/chainmcp/client.py, inside watch_revoked():\n")
print("\n".join(watch_src[i - 1 : i + 2]))
assert "daemon=True" in watch_src[i]

**✏️ Your turn 12.1 — predict the watcher's daemon flag**

Two predictions before running: (1) is `worker.daemon` — the toy thread above —
`True` or `False`? (2) how many lines of `watch_revoked`'s source (already captured
in `watch_src`) mention `daemon=`? Write both into the TODOs, run, then un-comment
the asserts. Success: both pass — the real watcher made the same choice your toy did.

In [ ]:
# TODO — predict BEFORE running:  worker.daemon is ???           (True / False)
# TODO — predict BEFORE running:  watch_revoked lines with "daemon=" = ???
daemon_lines = [l.strip() for l in watch_src if "daemon=" in l]
print("worker.daemon                  →", worker.daemon)
print('watch_revoked "daemon=" lines  →', len(daemon_lines))
for line in daemon_lines:
    print("   ", line)
# assert worker.daemon                # un-comment once confirmed
# assert len(daemon_lines) == 1       # un-comment once confirmed

<details><summary>✅ Solution 12.1 — peek only after trying</summary>

```python
assert worker.daemon
assert len(daemon_lines) == 1
```

Both hold — `True`, and exactly one line — because the daemon choice is made once,
right where the `Thread(...)` is built, so a finished program is never held open by
its background poller.
</details>

## 13. `from __future__ import annotations` — the first line of every repo file

Time to resolve section 4's puzzle: `fulfill`'s stored hints were the *strings* `'str'`
and `'int'`, not the types themselves. That's this import at work. It tells Python:
"don't evaluate type hints at definition time — store them as text." Two wins: a file
can hint with names defined later in the file (or too expensive to import at runtime),
and module import gets cheaper. Watch for: the toy function defined here (no future
import) storing real type objects, while all four repo modules store strings — and 4/4
carrying the line.

In [ ]:
def local_fn(x: int) -> str:   # this notebook has NO future import
    return str(x)

print("notebook fn hints →", local_fn.__annotations__, " ← real type objects")
print("repo fn hints     →", fakes.FakeChain.fulfill.__annotations__, " ← strings")

for mod in (models, ports, testing, fakes):
    first_lines = inspect.getsource(mod).splitlines()[:20]
    has_it = any(l.startswith("from __future__ import annotations") for l in first_lines)
    print(("✓" if has_it else "✗"), mod.__name__)
    assert has_it

**✏️ Your turn 13.1 — flip your own hints to strings**

`local_fn2` below stores real type objects, like any notebook function. Do the TODO —
add the future import as this cell's FIRST line — re-run, and watch the hints become
repo-style strings. Success: `repo-style?` flips to True and the assert passes.

In [ ]:
# TODO: make the FIRST line of this cell:  from __future__ import annotations
def local_fn2(x: int) -> str:
    return str(x)

print("local_fn2 hints →", local_fn2.__annotations__)
print("repo-style?     →", local_fn2.__annotations__ == {"x": "int", "return": "str"})
# assert local_fn2.__annotations__ == {"x": "int", "return": "str"}   # un-comment after the TODO

<details><summary>✅ Solution 13.1 — peek only after trying</summary>

```python
from __future__ import annotations

def local_fn2(x: int) -> str:
    return str(x)

assert local_fn2.__annotations__ == {"x": "int", "return": "str"}
```

The future import is a compiler directive scoped to a "file" — in a notebook, the cell
— telling Python to store hints as text instead of evaluating them (and IPython keeps
the flag for the rest of the session, so cells you run afterwards inherit it).
</details>

## What you learned (and where it goes)

This table is also your **"constructs I can now read" checklist** — every row is a
feature you'll meet again, in the notebook named on the right.

| You did | The construct | Where it becomes load-bearing |
|---|---|---|
| built `Ticket`, poked with `type`/`vars`/`dir`, desugared `self` | classes, instances, `__init__`, methods | every file in the repo |
| raised & caught `OfferAlreadyUsed` and friends | exceptions as named failures | the fakes (`05`), contract reverts (`06`, `07`) |
| rebuilt `RESOURCE_ID` with `f"{…:064x}"` | f-strings + format specs | ABI dissection (`04`) |
| pushed `'ha'` through an `int` hint | type hints = labels, not laws | pydantic enforcement (`02`) |
| proved `Optional[str] == Union[str, None] == str \| None` | `Union` / `\|` / `Optional` / `Literal` | discriminated unions (`02`) |
| read `Uint64` at the top of `models.py` | type aliases | every shape's fields (`02`) |
| unwrapped `Address` with `get_args` | `Annotated` sticky notes | pydantic reads the note (`02`) |
| wrote `@shout`, proved `@` = reassignment | decorators | `@runtime_checkable` (`03`), `@app.post` (`08`) |
| read `AnvilChain`, built a bogus one | dataclasses (they trust) | pydantic contrast (`02`), launching chains (`06`) |
| traced `BandwidthNeed → _Frozen → BaseModel` | inheritance, MRO | `_Frozen(BaseModel)` (`02`), ERC721 (`06`) |
| compared `Color` vs `SessionState` | `Enum` + `str` mixin | states & the deny vocabulary (`08`) |
| watched `Door.__exit__` survive an error; called `_free_port()` | context managers | disposable chains (`06`) |
| ran a `daemon=True` thread | `threading.Thread` | `watch_revoked` (`07`) |
| saw hints stored as strings | `from __future__ import annotations` | line 1 of every repo module |

## Experiments to try

Predict first, then run:

- Add a `transfer(self, new_owner)` method to `Ticket` that reassigns `self.owner` —
  predict what `vars(t7)` shows after `t7.transfer(fx.BELL)`, then check.
- Predict what `shout(shout(greet))()` returns (two wrappers deep), then run it.
- **Deliberate breakage:** in the `Door` class, change `return False` to `return True`
  in `__exit__`, re-run the raising cell, and predict what happens to the
  `RuntimeError`. You've just discovered exception *swallowing* — powerful, dangerous.
- Predict whether `SessionState("torn_down")` round-trips like `Color("red")` did; then
  try `SessionState("torn down")` (space, not underscore) and read the error carefully.

## Check yourself

1. `t7.describe()` — what is Python *really* calling, and what fills the `self` parameter?
2. Why does the repo define `OfferExpired` instead of raising `ValueError("expired")` everywhere?
3. `def f(x: int)` happily accepts `"hello"` — so what are hints *for*, and who in this
   project actually enforces them?
4. In your own words: what does the `@` in `@dataclass` do to the class beneath it?
5. Why does `SessionState` mix in `str` — what breaks without it?

**Where this goes next:** `02_pydantic_from_zero.ipynb` — the promises plain Python's
hints don't keep, pydantic keeps: every label you met today becomes an enforced gate at
the package borders.

**Deeper dive**

- [`e2e/notebooks/skeleton_explore.ipynb`](../skeleton_explore.ipynb) — these constructs used in anger, end to end
- [`e2e/notebooks/chain_client_explore.ipynb`](../chain_client_explore.ipynb) — `AnvilChain` and the watcher thread against a real chain
- [`docs/00-the-story.md`](../../../docs/00-the-story.md) — the story the code serves